 # Przykładowy problem 1

 Witaj. Tu znajdziesz opis przykładowego problemu dotyczącego rekomendacji (być może pośrednio reguł asocjacyjnych, a może masz inny pomysł?). Problem stanowi implementacja systemu oceniającego. Ten notebook pokaże w jaki sposób wykonywana jest analiza jakości systemu oceniającego. Zapoznajmy się z poszczególnymi elementami systemu.

 ### Oceny testowe

  Na ich podstawie wyznaczana będzie jakość systemu oceniającego. Ocen testowych BEZWZGLĘDNIE NIE MOŻNA BRAĆ POD UWAGĘ podczas tworzenia systemu. Oceny testowe, to po prostu losowo wybrane oceny spośród wszystkich ocen. Ten losowy wybór znajduje się w pliku test_users.py - jego treści nie powinieneś zmieniać.



In [ ]:
import test_users
import numpy as np
import csv
from tqdm import tqdm

test_users = test_users.test_users

 ### Filmy i użytkownicy

 Poniższe klasy reprezentują koncepty filmu i użytkownika. Zauważ, że struktury danych ograniczają się do prostych identyfikatorów filmów i ocen wystawionych przez użytkowników, dodatkowo dla obydwóch typów tworzone są indeksy. Ta implementacja **nie** może ulec zmianie. Jeżeli chcesz mieć dostęp do innych danych (np. do gatunków filmów, albo do opisów niepochodzących ze źródła danych), umieść implementacje źródeł danych w swoim systemie rekomendacyjnym, a nie tutaj.

In [ ]:
class Movie:
    index = {}
    name_index = {}
    inner_index = {}
    reverse_inner_index = {}
    inner_index_gen = 0
    def __init__(self, id, name):
        self.id = id
        self.name = name
        self.ratings = []
        self.genres = []
        self.genome_scores = {}
        Movie.index[id] = self
        Movie.name_index[name] = self
        
    def add_rating(self, rating):
        self.ratings.append(rating)
        
    def add_genome_score(self, tag_id, relevance):
        self.genome_scores[tag_id] = relevance
    
    
class User:
    index = {}
    def __init__(self, id):
        self.id = id
        self.ratings = {}
        User.index[id] = self
        
    def add_rating(self, movie, rating):
        movie.add_rating(rating)
        self.ratings[movie.id] = rating
        
    def __str__(self):
        str_bldr = f'{self.id}'
        return str_bldr


### Indeksy

Indeksy danych wypełniane są danymi z plików.


In [ ]:
with open('data/movie.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    csv_reader.__next__()
    for line in csv_reader:
        Movie(int(line[0]), line[1])

# Wczytywanie tagów (Genome Scores)
with open('data/genome_scores.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    csv_reader.__next__()
    for line in tqdm(csv_reader, total=11709768):
        movie_id = int(line[0])
        if movie_id in Movie.index:
            Movie.index[movie_id].add_genome_score(int(line[1]), float(line[2]))

with open('data/rating.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    csv_reader.__next__()
    for line in tqdm(csv_reader, total=20000263):
        if not int(line[0]) in User.index.keys():
            User(int(line[0]))
        User.index[int(line[0])].add_rating(Movie.index[int(line[1])],float(line[2]))

### EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")
print("EDA in progress... please hold\n")

all_relevance_scores = []
tags_per_movie = []
movies_with_tags_count = 0

for m_id, movie in Movie.index.items():
    if movie.genome_scores:
        movies_with_tags_count += 1
        all_relevance_scores.extend(movie.genome_scores.values())
        tags_per_movie.append(len(movie.genome_scores))

all_relevance_scores = np.array(all_relevance_scores)

print(f"# movies with tags: {movies_with_tags_count} (with {len(Movie.index)} total movies)")
if len(all_relevance_scores) > 0:
    print(f"# of loaded tag ratings: {len(all_relevance_scores)}")
    print(f"Average 'relevance' value: {np.mean(all_relevance_scores):.4f}")
    print(f"Median 'relevance' value: {np.median(all_relevance_scores):.4f}")
    print(f"Standard deviation: {np.std(all_relevance_scores):.4f}")

all_ratings = []
for u_id, user in User.index.items():
    all_ratings.extend(user.ratings.values())
all_ratings = np.array(all_ratings)

print(f"# of loaded user ratings: {len(all_ratings)}")
print(f"Average rating: {np.mean(all_ratings):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(all_relevance_scores, bins=50, color='skyblue', log=True, edgecolor='white')
axes[0].set_title('"Relevance" distribution of tag ratings')
axes[0].set_xlabel('Relevance (0.0 - 1.0)')
axes[0].set_ylabel('Frequency (log)')

sns.histplot(tags_per_movie, bins=30, ax=axes[1], color='salmon')
axes[1].set_title('# of tags assigned to each movie')
axes[1].set_xlabel('# of tags')
axes[1].set_ylabel('# of movies')

sns.histplot(all_ratings, bins=10, ax=axes[2], color='lightgreen', discrete=True)
axes[2].set_title('Distribution of movie ratings (1.0 - 5.0)')
axes[2].set_xlabel('Rating')
axes[2].set_ylabel('# of ratings')

plt.tight_layout()
plt.show()

movie_id_to_check = 1
if movie_id_to_check in Movie.index and Movie.index[movie_id_to_check].genome_scores:
    print(f"\nSANITY CHECK: '{Movie.index[movie_id_to_check].name}'")
    
    sorted_tags = sorted(Movie.index[movie_id_to_check].genome_scores.items(), key=lambda item: item[1], reverse=True)
    
    print("Top 5 most relevant tags:")
    for tag_id, relevance in sorted_tags[:5]:
        print(f" - Tag ID: {tag_id}, Relevance: {relevance:.4f}")
        
    print("\nTop 5 least relevant tags:")
    for tag_id, relevance in sorted_tags[-5:]:
        print(f" - Tag ID: {tag_id}, Relevance: {relevance:.4f}")
else:
    print("\nMovie not found or has no genome scores.")

 ### Systemy oceniające - klasa bazowa

 Każdy system oceniający ma dostęp do wszystkich użytkowników i ocen, które nie są ocenami użytkowników testowych. Można również zaimplementować metody doboru innych danych (np. gatunków, tagów, danych zewnętrznych). Poniżej znajduje się klasa bazowa - w jej inicjalizatorze wczytywane są dotyczące ocen wystawionych przez użytkowników nie będących testowymi.

In [ ]:
class RatingSystem:
    def __init__(self):
        self.users = {id : User.index[id] for id in User.index if id not in test_users}
        self.movie_ratings = {}
        for user in tqdm(self.users):
            for movie in self.users[user].ratings:
                if movie not in self.movie_ratings.keys():
                    self.movie_ratings[movie] = [self.users[user].ratings[movie]]
                else:
                    self.movie_ratings[movie].append(self.users[user].ratings[movie])
        
    def rate(self, user, movie):
        return


 ### Przykłady prostych systemów oceniających

 Poniższe przykłady są proste. Nie wykorzystują efektywnych rozwiązań (np. biblioteki numpy do obliczeń). Hipotezy, które te systemy realizują są bardzo naiwne. Twoim zadaniem jest napisanie systemu oceniającego, który będzie lepszy od tych systemów. Twój system będzie również porównany z systemami kolegów z grupy. Im wyższa pozycja w rankingu tym więcej punktów można uzyskać. Jeżeli zaimplementowany przez Ciebie system będzie lepszy niż system 'Naive Rating' otrzymasz 10 punktów; lepszy niż 'Average Global Movie Rating' - 12.5 punktów; lepszy niż wszystkie 4 przykładowe systemy - 17.5 punktów. Dodatkowo, jeżeli Twój system będzie jednym z najlepszych wśród implementacji kolegów z roku to otrzymasz:

 1. miejsce - 25 punktów
 2. miejsce - 22.5 punktów 
 3. miejsce - 20 punktów

 Przy czym, w przypadku wielu implementacji dających takie same wyniki, żeby otrzymać punkty za miejsce, system musi być lepszy, niż 6 najlepszy system (nie może remisować z systemem na pozycji 6).  
 
 Implementując swój system oceniający użyj następującej funkcji to string:

```
     def __str__(self):
        return 'Dowolna nazwa (MOJ NUMER INDEKSU)'
```

In [ ]:
class NaiveRating(RatingSystem):
    """
    Przykładowy system - naiwny. 
    Hipoteza: jeżeli zwrócę każdemu filmowi średnią ocenę (2.5/5), to moja ocena będzie niezła.
    """
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        return 2.5
    def __str__(self):
        return 'Naive Rating'

class AverageMovieRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        """
        Przykładowy system - średnia.
        Hipoteza: jeżeli zwrocę każdeu filmowi średnią ocenę (wynikającą z wszystkich ocen), to moja ocena będzie niezła.
        Jeżeli ten film jeszcze nie był oceniony, to zwrócę 2.5.
        """
        n = len(self.movie_ratings.get(movie, []))
        if n == 0:
            return 2.5
        else:
            return sum(self.movie_ratings[movie])/n
    def __str__(self):
        return 'Average Movie Rating'

class AverageUserRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        """
        Przykładowy system - średnia użytkownika.
        Hipoteza: jeżeli zwrócę dla tego filmu średnią ocenę wystawioną przez użytkownika, to mój system będzie niezły.
        """
        n = len(user.ratings.values())
        if n == 0:
            return 2.5
        else:
            return sum(user.ratings.values())/n
    def __str__(self):
        return 'Average User Rating'

class GlobalAverageMovieRating(RatingSystem):
    def __init__(self):
        """
        Przykładowy system - średnia ocena filmu.
        Hipoteza: średnia ocena tego filmu wśród wszystkich użytkowników powinna być dobrą estymacją.
        """
        super().__init__()
        self.GlobalAverageMovieRating = 0
        self.TotalMovies = 0
        for movie in self.movie_ratings:
            for rating in self.movie_ratings[movie]:
                self.GlobalAverageMovieRating += rating
                self.TotalMovies += 1
        if self.TotalMovies > 0:
            self.GlobalAverageMovieRating /= self.TotalMovies
        else:
            self.GlobalAverageMovieRating = 2.5

    def rate(self, user, movie):
        return self.GlobalAverageMovieRating
    def __str__(self):
        return 'Average Global Movie Rating'
    
class Cheater(RatingSystem):
    def __init__(self):
        super().__init__()

    def rate(self, user, movie):
        """
        Testowy system.
        Jeżeli ten system działa, to coś jest nie tak - systemy mają dostęp do ocen filmów, które mają wyznaczyć - powinien działać mniej więcej tak samo jak system naiwny.
        """
        if movie in user.ratings:
            return user.ratings[movie]
        else:
            return 2.5
    def __str__(self):
        return 'Cheater'


In [ ]:
class MySystem(RatingSystem):
    def __init__(self):
        super().__init__()
        self.movies = {id : Movie.index[id] for id in Movie.index}
        
    def calc_movie_avg(self, movie):
        ratings = self.movie_ratings.get(movie, [])
        n = len(ratings)
        if n == 0:
            return 2.5
        else:
            return sum(ratings)/n
            
    def calc_user_avg(self, user):
        n = len(user.ratings.values())
        if n == 0:
            return 2.5
        else:
            return sum(user.ratings.values())/n
            
    def rate(self, user, movie):
        """
        Ta metoda zwraca rating w skali 1-5. Jest to ocena przyznana przez użytkownika 'user' filmowi 'movie'.
        """
        movie_avg = self.calc_movie_avg(movie)
        
        rate_num = len(user.ratings)
        movies_rated_by_user = list(user.ratings.keys())
        random_sample = np.random.choice(movies_rated_by_user, 100, replace=False) if rate_num >= 100 else np.random.choice(movies_rated_by_user, rate_num, replace=False)
        
        offset = 0
        for m in random_sample:
            curr_movie = self.calc_movie_avg(m)
            offset += curr_movie - user.ratings[m]
            
        if len(random_sample) > 0:
            offset /= len(random_sample)
        else:
            offset = 0
            
        return movie_avg - offset

    def __str__(self):
        """
        Ta metoda zwraca numery indeksów wszystkich twórców rozwiązania. Poniżej przykład.
        """
        return 'System created by 156007 and 155833'

 ### Ocena systemów

 Poniższe klasy dotyczą oceny systemów - zwróć uwagę na parametr verbose, służy on do ograniczania informacji zwrotnej

In [ ]:
import copy
class RatingSystemCompetition:
    
    def __init__(self):
        self.registered_systems = []
        self.users = {id : User.index[id] for id in User.index if id not in test_users}
        self.verbose = 2
        
    def register(self, system):
        self.registered_systems.append(system)
        
    def build_round_robin(self):
        self.pairs = {}
        for system in self.registered_systems:
            self.pairs[system]  = []
            for competitor in self.registered_systems:
                if str(system) != str(competitor):
                    self.pairs[system].append((system, competitor))

            
    def runMatch(self, system, competitor):
        users_ids = np.random.choice(np.array(list(self.users.keys())), size=100)
        score = 0
        wins = 0
        loses = 0
        draws = 0
        for user_id in users_ids:
            user = self.users[user_id]
            user_copy = copy.deepcopy(self.users[user_id])
            movie_id = np.random.choice(np.array(list(user.ratings.keys())), size=1)[0]
            del user_copy.ratings[movie_id]
            true_rating = self.users[user_id].ratings[movie_id]
            system_rating = system.rate(user_copy,movie_id)
            competitor_rating = competitor.rate(user_copy,movie_id)
            
            if abs(true_rating - system_rating) <  abs(true_rating - competitor_rating):
                score += 1
                wins += 1
            elif abs(true_rating - system_rating) >  abs(true_rating - competitor_rating):
                score -= 1
                loses += 1
            else:
                draws += 1
                
        return score, wins, draws, loses
    
    def compete(self):
        self.total_scores = {}
        for system in self.pairs:
            self.total_scores[system] = 0
            if self.verbose >= 2: print(f'{system} analysis: ')
            for matchup in self.pairs[system]:
                score, wins, draws, loses = self.runMatch(matchup[0],matchup[1])
                if self.verbose >= 2: print(f'{matchup[0]} vs {matchup[1]} : {score} ({wins} wins, {draws} draws, {loses} loses)')
                self.total_scores[system] += score
            if self.verbose >= 2: print(f'{system} score: {self.total_scores[system]}')
        if self.verbose >= 1:
            print('Final scores: ')
            place = 1
            for system in sorted(self.total_scores, key=self.total_scores.get, reverse=True):
                print(f'{place}. {system}, {self.total_scores[system]} pkt')
                place += 1


 ### Przykładowa analiza

In [ ]:
competition = RatingSystemCompetition()
competition.register(MySystem())
competition.register(GlobalAverageMovieRating())
competition.register(NaiveRating())
competition.register(AverageMovieRating())
competition.register(AverageUserRating())
competition.register(Cheater())
competition.build_round_robin()
competition.compete()